In [1]:
import os
os.environ['HF_HOME'] = "/ssd_scratch/yuvraj.bodana/huggingface"
os.environ['HF_DATASETS_CACHE'] = "/ssd_scratch/yuvraj.bodana/huggingface"

In [2]:
from transformers import AutoModelForCausalLM,AutoTokenizer
from datasets import load_dataset
from peft import OFTConfig, LoraConfig,BOFTConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
import torch

In [3]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"
dataset_name = "ai4bharat/IN22-Gen"

lang_maps = {
    "Hindi":"hin_Deva",
    "Gujarati":"guj_Gujr",
    "Tamil":"tam_Taml",
    "Telugu":"tel_Telu",
    "Bengali":"ben_Beng",
    "English":"eng_Latn"
}

source_langs = ["English"]
target_langs = ["Gujarati", "Hindi", "Telugu", "Tamil", "Bengali"]

In [4]:
ds = load_dataset(dataset_name, split="test")

In [6]:
def make_prompts_batched(batch):
    all_messages = []
    
    # In batched mode, 'batch' is a dictionary of lists.
    # We find the number of rows in this specific batch.
    batch_size = len(batch[list(batch.keys())[0]])
    
    for i in range(batch_size):
        for src in source_langs:
            for tgt in target_langs:
                # Extract the sentences for the specific row 'i'
                src_sent = batch[lang_maps[src]][i]
                tgt_sent = batch[lang_maps[tgt]][i]

                prompt = f"""Translate following sentence from {src} to {tgt}:\n{src_sent}"""
                
                # Append one complete conversation per language pair
                all_messages.append([
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": tgt_sent}
                ])
                
    # Return the new column
    return {"messages": all_messages}

# CRITICAL: You must use batched=True AND remove the original columns
# so Hugging Face allows the total row count to change.
parallel_ds = ds.map(
    make_prompts_batched, 
    batched=True, 
    remove_columns=ds.column_names # This breaks the 1-to-1 row constraint
)

print(f"Original size: {len(ds)}")
print(f"New parallel size: {len(parallel_ds)}")

NameError: name 'ds' is not defined

In [6]:
parallel_ds = parallel_ds.shuffle(seed=42)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=torch.float16, device_map="auto",attn_implementation="eager"
)

In [6]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [9]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_rslora=True, # Using Rank Stabilized LoRA
    task_type="CAUSAL_LM",
    
    # To target all layers :
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],    
    
    ###Optimization###
    #use_dora=True,
    # init_lora_weights="gaussian",
    # modules_to_save=["lm_head"], 
    # ensure_weight_tying=True
)

In [10]:
lora_model = get_peft_model(model, lora_config)

In [11]:
lora_model.print_trainable_parameters()

trainable params: 22,544,384 || all params: 1,258,358,784 || trainable%: 1.7916


In [10]:
oft_config = OFTConfig(
    task_type="CAUSAL_LM",
    peft_type="OFT",
    r=32,
    oft_block_size=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],    
)

In [13]:
oft_model = get_peft_model(model, oft_config)

/scratch/yuvraj.bodana/myenv/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/scratch/yuvraj.bodana/myenv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [14]:
oft_model.print_trainable_parameters()

trainable params: 22,904,832 || all params: 1,258,719,232 || trainable%: 1.8197


In [11]:
boft_config = BOFTConfig(
    task_type="CAUSAL_LM",
    boft_block_size=4,
    boft_n_butterfly_factor=1,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    # boft_dropout=0.1
)

In [12]:
boft_model = get_peft_model(model, boft_config)

In [13]:
boft_model.print_trainable_parameters()

trainable params: 1,687,552 || all params: 1,237,501,952 || trainable%: 0.1364


In [11]:
sft_config = SFTConfig(
    output_dir="/ssd_scratch/yuvraj.bodana/FT_Model/",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=1,
    # save_steps=100,
    save_strategy="epoch",
    bf16=False,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    report_to="none",
    # save_total_limit=5,
    completion_only_loss=True,
    # max_length=4096,
    dataset_text_field="messages",
)

In [12]:
trainer = SFTTrainer(
    model=model,
    train_dataset=parallel_ds,
    processing_class=tokenizer,
    peft_config=oft_config,
    args=sft_config,
)

In [13]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
1,1.900300
2,1.279200
3,0.982400
4,0.972400
5,0.977700
6,0.985000
7,0.856400
8,0.849400
9,0.793300
10,0.910800


TrainOutput(global_step=640, training_loss=0.6996770747937262, metrics={'train_runtime': 1143.612, 'train_samples_per_second': 4.477, 'train_steps_per_second': 0.56, 'total_flos': 1.0925559294418944e+16, 'train_loss': 0.6996770747937262})

In [14]:
trainer.save_model('/ssd_scratch/yuvraj.bodana/FT_Model/')

In [21]:
test_data = load_dataset("ai4bharat/IN22-Conv", split='test')

In [24]:
list(test_data['eng_Latn'])

["Mom, let's go for a movie tomorrow.",
 "I don't have to go to school.",
 'It is a holiday.',
 'Oh, tomorrow is the 14th of April right?',
 'Your dad will also have the day off from work.',
 'We can make a movie plan!',
 "That's a good news!",
 'Why is it a holiday though?',
 'Are all schools, colleges and offices closed tomorrow?',
 'It is Ambedkar Jayanti tomorrow!',
 'This day is celebrated annually to mark the birth of Dr. B. R Ambedkar.',
 'Have you heard of him?',
 'I think I have seen him in my History and Civics book.',
 'Is he related to our Constitution?',
 'Absolutely! He is known as the father of the Indian Constitution.',
 'He was a civil rights activist who played a major role in formulating the Constitution.',
 'He played a crucial part in shaping the vibrant democratic structure that India prides itself upon.',
 'I remember now!',
 'We were discussing about him in our Civics class the other day.',
 "Ma'am said that she will tell us more about him when we do the chapter

In [7]:
def make_prompts_batched(batch):
    all_messages = []
    
    # In batched mode, 'batch' is a dictionary of lists.
    # We find the number of rows in this specific batch.
    batch_size = len(batch[list(batch.keys())[0]])
    
    for i in range(batch_size):
        for src in source_langs:
            for tgt in target_langs:
                # Extract the sentences for the specific row 'i'
                src_sent = batch[lang_maps[src]][i]
                tgt_sent = batch[lang_maps[tgt]][i]

                prompt = f"""Translate following sentence from {src} to {tgt}:\n{src_sent}"""
                
                # Append one complete conversation per language pair
                all_messages.append([
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": tgt_sent}
                ])
                
    # Return the new column
    return {"messages": all_messages}

In [9]:
test_data = test_data.map(make_prompts_batched, batched=True, remove_columns=test_data.column_names)

Map:   0%|          | 0/1503 [00:00<?, ? examples/s]

In [12]:
test_data['messages'][0:20]

[[{'role': 'user',
   'content': "Translate following sentence from English to Gujarati:\nMom, let's go for a movie tomorrow."},
  {'role': 'assistant',
   'content': 'મમ્મી, ચાલો આપણે આવતીકાલે ફિલ્મ જોવા જઇએ.'}],
 [{'role': 'user',
   'content': "Translate following sentence from English to Hindi:\nMom, let's go for a movie tomorrow."},
  {'role': 'assistant', 'content': 'माँ, चलो कल एक फिल्म देखने चलते हैं।'}],
 [{'role': 'user',
   'content': "Translate following sentence from English to Telugu:\nMom, let's go for a movie tomorrow."},
  {'role': 'assistant', 'content': 'అమ్మా, రేపు సినిమాకి వెళ్దాం.'}],
 [{'role': 'user',
   'content': "Translate following sentence from English to Tamil:\nMom, let's go for a movie tomorrow."},
  {'role': 'assistant',
   'content': 'அம்மா, நாம் நாளைக்கு சினிமா பார்க்கப் போகலாமா?'}],
 [{'role': 'user',
   'content': "Translate following sentence from English to Bengali:\nMom, let's go for a movie tomorrow."},
  {'role': 'assistant', 'content': 'মা, চল

In [13]:
test_data.to_csv("./test_data.tsv", sep='\t')

Creating CSV from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

2439947

In [18]:
test_data['messages'][0]

[{'role': 'user',
  'content': "Translate following sentence from English to Gujarati:\nMom, let's go for a movie tomorrow."},
 {'role': 'assistant', 'content': 'મમ્મી, ચાલો આપણે આવતીકાલે ફિલ્મ જોવા જઇએ.'}]

In [25]:
import pandas as pd

In [26]:
df = pd.read_csv('./OFT_Results/English_Gujarati.tsv', sep='\t')

In [28]:
for i, row in df.iterrows():
    print(row.Source)
    print(row.MT)
    print(row.Target)
    print("===")

Mom, let's go for a movie tomorrow.
માતા, આજની રોજ ફિલ્મ કહેવાનો જરૂરી છે.
મમ્મી, ચાલો આપણે આવતીકાલે ફિલ્મ જોવા જઇએ.
===
I don't have to go to school.
મને શાળામાં જવાની જરૂરિયાત નથી.
મારે કાલે શાળાએ જવાનું નથી.
===
It is a holiday.
તે ઉત્સવ છે.
રજા છે.
===
Oh, tomorrow is the 14th of April right?
હું ત્યારે ૧૪મી એપ્રિલ હોવા છું?
ઓહ, આવતીકાલે 14મી એપ્રિલ છે, બરાબર?
===
Your dad will also have the day off from work.
તેના પિતાને પણ કારકરીની રોજગારી થઈ શકશે.
તારા પપ્પાને પણ તેમના કામમાંથી એક દિવસની રજા હશે.
===
We can make a movie plan!
આમાં એક ફિલ્મ પરિણામ કરવાનું જરૂરી છે!
આપણે ફિલ્મ જોવાની યોજના બનાવી શકીએ!
===
That's a good news!
આ સારું વાત છે!
આ તો સારા સમાચાર છે!
===
Why is it a holiday though?
કારણ કે કોઈ હાલની ઉત્સવ છે કે નહીં?
પણ આવતીકાલે શા માટે રજા છે?
===
Are all schools, colleges and offices closed tomorrow?
દિવસના રોજ કોઈ શાળા, કોલેજ અને કાર્યાલય બંધ થયાં હોવાનું કહેવાય છે?
બધી શાળાઓ, કોલેજો અને કાર્યાલયો પણ આવતીકાલે બંધ હશે?
===
It is Ambedkar Jayanti tomorrow!
તેની આજે અમે

In [29]:
df_base = pd.read_csv('./Base_Results/English_Gujarati.tsv', sep='\t')

In [30]:
for i, row in df_base.iterrows():
    print(row.Source)
    print(row.MT)
    print(row.Target)
    print("===")

Mom, let's go for a movie tomorrow.
મારી માતા, આખરે એક ફિલ્મ દરમિયાન જઈને જાઓ.
મમ્મી, ચાલો આપણે આવતીકાલે ફિલ્મ જોવા જઇએ.
===
I don't have to go to school.
મારી શાળામાં જવાની જરૂર નથી.
મારે કાલે શાળાએ જવાનું નથી.
===
It is a holiday.
આ સંદર્ભમાં સંદર્ભમાં એક ઉત્સવ તરીકે માનવામાં આવે છે.
રજા છે.
===
Oh, tomorrow is the 14th of April right?
આખરી તારીખ એટલે ૧૪ એપ્રિલ છે.
ઓહ, આવતીકાલે 14મી એપ્રિલ છે, બરાબર?
===
Your dad will also have the day off from work.
તમારા પિતાને પણ કામમાં રહેવાની દિવસીય સમયગાળી પણ તેમને કામમાં રહેવાની દિવસીય સમયગાળી પણ આપણે તેમને દિવસે રહેવાની દિવસીય સમયગાળી પણ આપી શકો છો.
તારા પપ્પાને પણ તેમના કામમાંથી એક દિવસની રજા હશે.
===
We can make a movie plan!
મારી ફિલ્મ પરિકલ્પના માટે આપણે કઈ કરીએ?
આપણે ફિલ્મ જોવાની યોજના બનાવી શકીએ!
===
That's a good news!
આ એક સુખાકાર વાર્તા છે.
આ તો સારા સમાચાર છે!
===
Why is it a holiday though?
તે એક હિન્દુ ઉત્સવ છે. તેની સંદર્ભકર્તા હિન્દુ ધર્મની પ્રથમ વિશેષતા છે.
પણ આવતીકાલે શા માટે રજા છે?
===
Are all schools, colleges and offices c

In [31]:
df_lora = pd.read_csv('./Lora_Results/English_Gujarati.tsv', sep='\t')

In [32]:
for i, row in df_lora.iterrows():
    print(row.Source)
    print(row.MT)
    print(row.Target)
    print("===")

Mom, let's go for a movie tomorrow.
માતા, આજ રોજ એક ફિલ્મ કહો છો.
મમ્મી, ચાલો આપણે આવતીકાલે ફિલ્મ જોવા જઇએ.
===
I don't have to go to school.
મને શાળામાં જવા જ પડતું નથી.
મારે કાલે શાળાએ જવાનું નથી.
===
It is a holiday.
તે એક ઉત્સવ છે.
રજા છે.
===
Oh, tomorrow is the 14th of April right?
હું તો આજે મારી જાણ છું કે આજે ૧૪મી એપ્રિલ છે?
ઓહ, આવતીકાલે 14મી એપ્રિલ છે, બરાબર?
===
Your dad will also have the day off from work.
તેના પિતાને પણ કાર્યક્રમથી એક દિવસ ખરોઈ જવાનું હોય તે પણ હોય.
તારા પપ્પાને પણ તેમના કામમાંથી એક દિવસની રજા હશે.
===
We can make a movie plan!
મારી એક ફિલ્મ કાર્યરત કરવા માટે આપણે એક ફિલ્મ કાર્યરત કરી શકીએ!
આપણે ફિલ્મ જોવાની યોજના બનાવી શકીએ!
===
That's a good news!
આ એ વધુ સુખાય છે!
આ તો સારા સમાચાર છે!
===
Why is it a holiday though?
કારણ કે કે કોઈ કારણ નથી કે?
પણ આવતીકાલે શા માટે રજા છે?
===
Are all schools, colleges and offices closed tomorrow?
કોઈ શાળા, કોલેજ અને કાર્યાલય પહેલાં રહેલા છે?
બધી શાળાઓ, કોલેજો અને કાર્યાલયો પણ આવતીકાલે બંધ હશે?
===
It is Ambedkar Jayanti

In [34]:
from comet import download_model, load_from_checkpoint
from pathlib import Path

In [35]:
dataset_path = Path("./Base_Results")
tsv_files = list(dataset_path.glob("*.tsv"))

In [36]:
tsv_files

[PosixPath('Base_Results/English_Hindi.tsv'),
 PosixPath('Base_Results/English_Tamil.tsv'),
 PosixPath('Base_Results/English_Gujarati.tsv'),
 PosixPath('Base_Results/English_Bengali.tsv'),
 PosixPath('Base_Results/English_Telugu.tsv')]

In [45]:
def example(row):
    # print(row)
    print(row['Source'])

In [46]:
df.apply(example)

KeyError: 'Source'